# Code for doing the different clustering methods, as well as data preprocessing

In this file we run the different methods

### DMACN - Deep Multi-kernel Auto-encoder Clustering Network
First load the data

In [ ]:
# Loading the data with all 400 features (for 400x400 data)

from scipy.io import loadmat
import torch
import numpy as np

FC_test_mat = loadmat("C:\\Mats og Odd Arne\\Prosjektoppgave\\sch407\\YA\\test_vectorized_fc.mat")

FC_test_array = FC_test_mat["vectorized_fc"]  # Example matrix
# np.fill_diagonal(FC_test_array, 1.0)  # Set diagonal to zero

print(FC_test_array[0:5].shape)  # Print the first 5 rows to verify

X = torch.from_numpy(FC_test_array[0:5]).float()  # Example matrix

Define the autoencoder specs

In [ ]:
from DMACN import DMACN, DMACNConfig

kernel_specs = [
    {"kind": "rbf", "t": 0.01},
    {"kind": "rbf", "t": 0.05},
    {"kind": "rbf", "t": 0.1},
    {"kind": "rbf", "t": 1},
    {"kind": "rbf", "t": 10},
    {"kind": "rbf", "t": 50},
    {"kind": "rbf", "t": 100},
    {"kind": "poly", "a": 0, "b": 2},
    {"kind": "poly", "a": 0, "b": 4},
    {"kind": "poly", "a": 1, "b": 2},
    {"kind": "poly", "a": 1, "b": 4}
]  # h = 3


# Config for 29x29 data
cfg = DMACNConfig(
    C=3,  # number of clusters
    dims_enc=[79800, 56427, 39900],   # N / 2^(i/l). l=number of layers, N=number of features 
    dims_dec=[39900, 56427, 79800],
    kernel_specs=kernel_specs,
    m_fuzz=1.08,
    lam1=0.5,
    lam2=0.5,
    lr=1e-3,
    epochs=500,
    mk_max_iters=20,
    mk_eps_stop=1e-5,
    renormalize_omega_sum1=True,
    mid_only_first=True,
    mid_only_last=True,
)


Run the model

In [ ]:
model = DMACN(cfg)
model.fit(X, verbose_every=50)
labels = model.predict(save=True)
print("labels shape:", labels.shape)
print("labels: ", labels)

### UMAP

In [ ]:
from UMAP import UMAP

### HDBSCAN
Perform HDBSCAN on the data

In [ ]:
from HDBSCAN import hdbscan_clustering

hdbscan_clustering(Functional_connectivity_matrix=Functional_connectivity_matrix, save_labels=True)

### Evaluate the clusters
Evaluate the clusters using simple methods: Silhouette coefficient, Davies-Bouldin score and Calinski-Harabasz score

In [ ]:
from Evaluate_models import evaluate_clustering
import os

# Define where to find the labels 
labels_path = os.fsencode("Clusters")
evaluate_clustering(functional_connectivity_matrix=Functional_connectivity_matrix, labels_path=labels_path)